# 持久化模式

上面讲了怎么配置检查点存储

这里另一方面， 什么时候写入，
1. 写入越及时，恢复的状态就越完整，容灾能力越强，什么时候写入，也是有说法的， 写入越频繁，越及时，比如每1s保存， 恢复前1s就行，但是1h写一次1h写一次，恢复的就不完整
2. 但写的太及时，要执行更多中间段的写入，比如1s写一次，每次都要间隔去写， 并且如果设置同步模式还要去等待，有额外的性能开销和相应样式


有三种持久化模式
1. exit 非常不及时， 平常不写检查点， 只有图结束了，运行结束了或中断了，写检查点，  如果是因为异常退出，只能退出之后记录检查点， 但是如果程序崩溃，也不能正常写入， 相应不及时，性能开销小，容灾能力最弱
2. async  异步模式， 每个superstep结束后， 写完整的检查点，并且再图中任务执行完毕后，记录中间结果，  每个超步写一次检查点，性能开销更大，写入操作再后台， 并行执行，A-》B ， 箭头中间执行内容， 不影响主流程B的运行时间 （默认和推荐使用的模式）
3. sync：同步模式， 写入的时间点也是 一个superstep结束后，开始写检查点，相应及时，  但是最大区别是 在下一个supterstep之前要等待卡死 写入再执行B， 有一点额外的时间响应延时，但是容灾能力最强， 因为可能异步模式， A-》B写入的时候 进程B崩了，导致B前异步写入没有写下来，   异步模式比同步模式弱一个超步， 浪费一点时间

示例不去演示 崩了的情况

因为真正蹦只有 断电情况（进程崩溃， 还有没有办法保存的异常才能体现差异）， 一个图如果能正常运行 网络通讯、retry满了 抛出异常也是能记录检查点的，区别不大， 只有真正崩溃才有区别


不太多演示， 如果想改的话
 
 在调用任务的时候
 ```python
 # 调用时传递
res=graph.invoke(
    {"messages": [HumanMessage("你好")]},
    config=config,
    durability="async" # sync / exit  改成三种模式， 默认是异步模式，通常使用异步模式
)
print(res["output"])
```

异步模式不会有延迟感觉，也会用一点点性能， 在每个超步运行完之后，去写入